# Q4-Q5 - Fresh inference and genuine errors
The same selected checkpoint and preprocessing serve the API and Grad-CAM.

In [ ]:
from pathlib import Path
import sys, json
PROJECT = Path.cwd()
if PROJECT.name == 'notebooks': PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / 'src'))
from fabric_inspector.common import read_json
from IPython.display import display, Image as DisplayImage, Markdown
import pandas as pd
def read(path): return read_json(PROJECT / path)
def figure(path): display(DisplayImage(filename=str(PROJECT / path)))


In [ ]:
import torch, os
os.environ['FABRIC_CPU_THREADS']='1'
torch.set_num_threads(1)
from fabric_inspector.serving.predictor import Predictor
from fabric_inspector.data.manifest import load_manifest
predictor=Predictor(PROJECT/'artifacts/serving-bundle.json')
sample,rows,root=load_manifest(PROJECT/'data/sample-manifests/split.json',verify_files=True)
row=next(r for r in rows if r['split']=='val')
result=predictor.predict((root/row['image_path']).read_bytes())
display(result)

In [ ]:
from PIL import Image
from fabric_inspector.data.images import preprocess
from fabric_inspector.explainability.gradcam import gradcam,overlay,occlusion_check
image=Image.open(root/row['image_path']).convert('L')
tensor=preprocess(image,predictor.metadata['preprocessing']).unsqueeze(0).to(predictor.device)
cam=gradcam(predictor.model,tensor)
assert cam.shape==(256,256)
display(overlay(image,cam))
display(occlusion_check(predictor.model,tensor,cam))

In [ ]:
cases=read('docs/three-xai-errors.json')['cases']
assert len(cases)>=3 and all(c['incorrect'] and c['label']!=c['prediction'] for c in cases)
for case in cases:
    display(Markdown('**'+case['case_name']+'** - '+case['analysis']+'\n\nVisual inspection: '+case['visual_analysis']))
    figure(case['panel'])
    display(case['occlusion'])

In [ ]:
verification=read('reports/application-verification/results.json')
assert verification['status']=='PASS'
display(verification)

Grad-CAM is a coarse defect-logit sensitivity map; it is not a segmentation output or causal proof. Equal-area top/low/random mean-fill occlusion tests sensitivity. Negative drops, zero CAMs and failures against controls are reported. Integrated Gradients depends on a reference image; LIME depends on perturbations and a local surrogate.